In [1]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Customer_Satisfaction_PCA').getOrCreate()
from sklearn.model_selection import train_test_split
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from sklearn.decomposition import PCA

In [2]:
data = pd.read_csv("preprocessed_data.csv")
data.drop("Unnamed: 0",axis=1,inplace=True)
data.head(5)

,payment_type,payment_installments,payment_value,review_score,customer_state,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,...,seller_state,distance,delivery_days,estimated_days,ships_in,review_time,arrival_time,delivery_impression,estimated_del_impression,ship_impression
0,3,8,99.33,1,0,921.0,8.0,800.0,17.0,27.0,...,0,845.34,14,27,7,5,1,2,0,2
1,3,1,24.39,5,1,1274.0,2.0,150.0,16.0,6.0,...,0,23.27,3,20,6,3,1,2,1,2
2,3,1,65.71,5,1,1536.0,2.0,250.0,20.0,8.0,...,0,27.31,6,23,14,3,1,2,1,1
3,3,8,107.78,5,0,188.0,1.0,1200.0,44.0,2.0,...,0,457.50,15,29,6,0,1,2,0,2
4,3,8,107.78,5,0,188.0,1.0,1200.0,44.0,2.0,...,0,457.50,15,29,6,1,1,2,0,2


In [3]:
X_train, X_test, y_train, y_test = train_test_split(data.drop('review_score', axis=1), data['review_score'], test_size=0.2, random_state=42)

In [4]:
pca = PCA(n_components=4)

components = pca.fit_transform(X_train)
labels = {
    str(i): f"PC {i+1} ({var:.1f}%)"
    for i, var in enumerate(pca.explained_variance_ratio_ * 100)
}

In [5]:
components = components[:, :4]
labels = [f"PC {i+1}" for i in range(components.shape[1]) ]

components = pd.DataFrame(data=components, columns=labels)

components['review_score'] = data['review_score']

components.to_csv('train_data_pca.csv', index=False)

In [6]:
pca = PCA(n_components=4)

components = pca.fit_transform(X_test)
labels = {
    str(i): f"PC {i+1} ({var:.1f}%)"
    for i, var in enumerate(pca.explained_variance_ratio_ * 100)
}

In [7]:
components = components[:, :4]
labels = [f"PC {i+1}" for i in range(components.shape[1]) ]

components = pd.DataFrame(data=components, columns=labels)

components['review_score'] = data['review_score']

components.to_csv('test_data_pca.csv', index=False)

In [8]:
train_data = spark.read.csv("train_data_pca.csv", header=True, inferSchema=True)
test_data = spark.read.csv("test_data_pca.csv", header=True, inferSchema=True)

In [9]:
cols = ['PC 1', 'PC 2', 'PC 3', 'PC 4']

In [10]:
train_assembler = VectorAssembler(inputCols=cols, outputCol='features')
transformed_train_data = train_assembler.transform(train_data)
transformed_train_data.show(5, truncate=False)

test_assembler = VectorAssembler(inputCols=cols, outputCol='features')
transformed_test_data = test_assembler.transform(test_data) 
transformed_test_data.show(5, truncate=False)

+-------------------+-------------------+-------------------+-------------------+------------+-------------------------------------------------------------------------------+
|PC 1               |PC 2               |PC 3               |PC 4               |review_score|features                                                                       |
+-------------------+-------------------+-------------------+-------------------+------------+-------------------------------------------------------------------------------+
|-1655.9086923920459|155.20380726947727 |-590.1757779520456 |-78.44429481979748 |1           |[-1655.9086923920459,155.20380726947727,-590.1757779520456,-78.44429481979748] |
|449.7553511617574  |-253.27793533016722|-409.25098942327804|355.38052776880295 |5           |[449.7553511617574,-253.27793533016722,-409.25098942327804,355.38052776880295] |
|-1206.1920317348608|271.770083833671   |-274.9076043003584 |-123.68124957482584|5           |[-1206.1920317348608,271.770083

In [11]:
train_data = transformed_train_data.select('features', 'review_score')
train_data.show(5, truncate=False)

test_data = transformed_test_data.select('features', 'review_score') 
test_data.show(5, truncate=False)

+-------------------------------------------------------------------------------+------------+
|features                                                                       |review_score|
+-------------------------------------------------------------------------------+------------+
|[-1655.9086923920459,155.20380726947727,-590.1757779520456,-78.44429481979748] |1           |
|[449.7553511617574,-253.27793533016722,-409.25098942327804,355.38052776880295] |5           |
|[-1206.1920317348608,271.770083833671,-274.9076043003584,-123.68124957482584]  |5           |
|[-451.32142274711805,-583.5820932180875,38.889318844342135,10.73583525726629]  |5           |
|[-1805.1357686157887,56.180853001162795,-620.2705802510933,-0.7730618395956845]|5           |
+-------------------------------------------------------------------------------+------------+
only showing top 5 rows

+-----------------------------------------------------------------------------+------------+
|features                  

In [12]:
# Random Forrest

clf = RandomForestClassifier(labelCol='review_score', featuresCol='features')
clf = clf.fit(train_data)

predictions = clf.transform(test_data)
predictions.show(5, truncate=False)
predictions.select('review_score', 'prediction').show(5, truncate=False)

evaluator_multi = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='accuracy')
evaluator_weighted_precision = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='weightedPrecision')
evaluator_weighted_recall = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='weightedRecall')
evaluator_f1 = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='f1')

accuracy = evaluator_multi.evaluate(predictions)
weighted_precision = evaluator_weighted_precision.evaluate(predictions)
weighted_recall = evaluator_weighted_recall.evaluate(predictions)
f1 = evaluator_f1.evaluate(predictions)

print(f'Accuracy: {accuracy}')
print(f'Weighted Precision: {weighted_precision}')
print(f'Weighted Recall: {weighted_recall}')
print(f'F1 Score: {f1}')

+-----------------------------------------------------------------------------+------------+----------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------+----------+
|features                                                                     |review_score|rawPrediction                                                                                       |probability                                                                                              |prediction|
+-----------------------------------------------------------------------------+------------+----------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------+----------+
|[6979.555900910083,243.69306521528566,333.9769433368753,-157.61300

In [13]:
# Logistic Regression

clf = LogisticRegression(labelCol='review_score', featuresCol='features')
clf = clf.fit(train_data)

predictions = clf.transform(test_data)
predictions.show(5, truncate=False)
predictions.select('review_score', 'prediction').show(5, truncate=False)

evaluator_multi = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='accuracy')
evaluator_weighted_precision = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='weightedPrecision')
evaluator_weighted_recall = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='weightedRecall')
evaluator_f1 = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='f1')

accuracy = evaluator_multi.evaluate(predictions)
weighted_precision = evaluator_weighted_precision.evaluate(predictions)
weighted_recall = evaluator_weighted_recall.evaluate(predictions)
f1 = evaluator_f1.evaluate(predictions)

print(f'Accuracy: {accuracy}')
print(f'Weighted Precision: {weighted_precision}')
print(f'Weighted Recall: {weighted_recall}')
print(f'F1 Score: {f1}')

+-----------------------------------------------------------------------------+------------+------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------+----------+
|features                                                                     |review_score|rawPrediction                                                                                                     |probability                                                                                                                |prediction|
+-----------------------------------------------------------------------------+------------+------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------

In [14]:
# Decision Tree 

clf = DecisionTreeClassifier(labelCol='review_score', featuresCol='features')
clf = clf.fit(train_data)

predictions = clf.transform(test_data)
predictions.show(5, truncate=False)
predictions.select('review_score', 'prediction').show(5, truncate=False)

evaluator_multi = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='accuracy')
evaluator_weighted_precision = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='weightedPrecision')
evaluator_weighted_recall = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='weightedRecall')
evaluator_f1 = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='review_score', metricName='f1')

accuracy = evaluator_multi.evaluate(predictions)
weighted_precision = evaluator_weighted_precision.evaluate(predictions)
weighted_recall = evaluator_weighted_recall.evaluate(predictions)
f1 = evaluator_f1.evaluate(predictions)

print(f'Accuracy: {accuracy}')
print(f'Weighted Precision: {weighted_precision}')
print(f'Weighted Recall: {weighted_recall}')
print(f'F1 Score: {f1}')

+-----------------------------------------------------------------------------+------------+------------------------------------------+-------------------------------------------------------------------------------------------------------+----------+
|features                                                                     |review_score|rawPrediction                             |probability                                                                                            |prediction|
+-----------------------------------------------------------------------------+------------+------------------------------------------+-------------------------------------------------------------------------------------------------------+----------+
|[6979.555900910083,243.69306521528566,333.9769433368753,-157.6130037542189]  |1           |[0.0,2353.0,707.0,1700.0,3936.0,11126.0]  |[0.0,0.11870648774089396,0.03566744021793966,0.08576329331046312,0.1985672485117546,0.5612955302189486]|5.0     